# 05 Test results
Latest full run per model, split by training audio (raw vs trimmed). EER recomputed from each run's score files.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from audio_df.helpers.research.runs import collect, eer, eer_table

MODELS = ["a1", "a4", "a2", "d1", "d2", "d3"]
results = collect("../runs", MODELS)
pd.DataFrame({k: {"artifact": v["artifact"], "runs": ", ".join(v["runs"])} for k, v in results.items()}).T

## EER % per test set

In [ ]:
table = eer_table(results)
table.loc[[(m, t) for m in MODELS for t in ["raw", "trimmed"] if (m, t) in table.index]].round(2)

## Per attack, eval (EER %)

In [ ]:
def per_attack(df):
    bona = df[df.label == "bonafide"]
    return {a: eer(pd.concat([bona, g])) for a, g in df[df.label == "spoof"].groupby("attack_id")}

pd.DataFrame({k: per_attack(v["scores"]["eval"]) for k, v in results.items()}).T.round(1)

## Score distributions
Higher = bona fide. A model that collapses puts both classes on the same side.

In [ ]:
sets = ["dev", "eval", "itw"]
fig, axes = plt.subplots(len(results), len(sets), figsize=(12, 1.8 * len(results)), squeeze=False)
for i, (k, v) in enumerate(results.items()):
    v = v["scores"]
    for j, t in enumerate(sets):
        ax = axes[i, j]
        for lab, color in [("bonafide", "tab:green"), ("spoof", "tab:orange")]:
            ax.hist(v[t].score[v[t].label == lab], bins=60, density=True, histtype="step", color=color, label=lab)
        ax.set_title(f"{k[0]} ({k[1]} train) - {t}", fontsize=9)
        ax.set_yticks([])
axes[0, 0].legend(fontsize=8)
fig.tight_layout()
plt.show()